**Adverserial resistant Fake payment Gateway detection using transactions patterns
This project detects fake payment gateways using URL metadata, transaction patterns, and adversarial-resistant feature engineering.**

**Data sets Collection (Without Adverserial)**
#
Adverserial is for fooling a model as attacker can be able to bypass

In [1]:
import pandas as pd
df = pd.read_csv('../raw/adverserial_legit.csv')
print(df)

                           url  label  TransactionAmount TransactionType  \
0            http://skrill.com      1               0.49           Debit   
1            http://stripe.com      0             120.50           Debit   
2    https://bankofamerica.com      1               9.99           Debit   
3            http://skrill.com      1               0.49           Debit   
4         http://apple pay.com      1               1.99           Debit   
..                         ...    ...                ...             ...   
995  https://bankofamerica.com      1               9.99           Debit   
996          http://paypal.com      1               4.99           Debit   
997          http://stripe.com      0             120.50           Debit   
998           http://wepay.com      0              75.00          Credit   
999       http://apple pay.com      1               1.99           Debit   

     CustomerAge  AccountBalance    Location  
0             19           150.0  Chitta

**Adverserial Generations**
#
interms of there homoglyph_domain,add_https,mimic_timing,age_spoof

In [2]:
import pandas as pd
import random
from urllib.parse import urlparse
import os


def homoglyph_domain(url):
    p = urlparse(url)
    host = p.netloc
    host2 = host.replace('l','I').replace('o','0').replace('a','@')
    return url.replace(host, host2)

def add_https(url):
    if url.startswith('http://'):
        return url.replace('http://', 'https://')
    return url

def mimic_timing(row):
    return random.uniform(5.0, 15.0)

def age_spoof(row):
    return random.choice([15, 30, 60, 90])

def generate_variants(df, n_variants=5):
    adv_rows = []

    phish_targets = df[df['label'] == 1]

    required_cols = list(df.columns) + [
        'notes', 'num_redirects', 'https_flag',
        'domain_age_days', 'time_to_confirm', 'page_dwell'
    ]

    for _, row in phish_targets.iterrows():
        for i in range(n_variants):
            r = row.copy()

            
            for col in ['notes', 'num_redirects', 'https_flag',
                        'domain_age_days', 'time_to_confirm', 'page_dwell']:
                r[col] = None

            attack = random.choice([
                'homoglyph', 'https', 'timing',
                'age', 'redirects', 'amount_mimic'
            ])

            if attack == 'homoglyph':
                r['url'] = homoglyph_domain(r['url'])
                r['notes'] = 'homoglyph'

            elif attack == 'https':
                r['url'] = add_https(r['url'])
                r['https_flag'] = 1
                r['notes'] = 'https_added'

            elif attack == 'timing':
                r['time_to_confirm'] = mimic_timing(row)
                r['page_dwell'] = max(0.5, random.uniform(0.7, 1.5))
                r['notes'] = 'timing_mimic'

            elif attack == 'age':
                r['domain_age_days'] = age_spoof(row)
                r['notes'] = 'age_spoof'

            elif attack == 'redirects':
                r['num_redirects'] = random.randint(3, 7)
                r['notes'] = 'more_redirects'

            elif attack == 'amount_mimic':
                r['TransactionAmount'] = random.choice([0.49, 1.99, 5.0, 9.99])
                r['notes'] = 'amount_mimic'

            adv_rows.append(r)

    adv_df = pd.DataFrame(adv_rows)
    adv_df = adv_df.reindex(columns=required_cols)

    
    
    adv_df = adv_df.dropna(axis=1, how='any')

    
    adv_df = adv_df.dropna(axis=0, how='any')
    

    return adv_df


if __name__ == "__main__":
    os.makedirs('../processed', exist_ok=True)
    os.makedirs('../raw', exist_ok=True)

    input_path = "../raw/adverserial_legit.csv"
    output_path = "../processed/sessions_with_adv.csv"

    try:
        manual_df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: {input_path} not found. Please create it with the updated structure first.")
        exit()

    print(f"Generating adversarial variants from {len(manual_df[manual_df['label'] == 1])} targets...")

    adv_test_df = generate_variants(manual_df)

    adv_test_df.to_csv(output_path, index=False)

    print(f"Saved {len(adv_test_df)} adversarial test samples to {output_path}")


Generating adversarial variants from 490 targets...
Saved 2450 adversarial test samples to ../processed/sessions_with_adv.csv


**Dataset(Adverserial Generated)** 
#
Adverserial dataset created for model so that model understand adverserial datasets 


In [3]:
import pandas as pd
df = pd.read_csv('../processed/sessions_with_adv.csv')
print(df.head())

                  url  label  TransactionAmount TransactionType  CustomerAge  \
0   http://paypal.com      1               5.00           Debit           35   
1   http://paypal.com      1               4.99           Debit           35   
2   http://paypal.com      1               4.99           Debit           35   
3   http://paypal.com      1               1.99           Debit           35   
4  https://paypal.com      1               4.99           Debit           35   

   AccountBalance Location         notes  
0          1200.0   Sylhet  amount_mimic  
1          1200.0   Sylhet  timing_mimic  
2          1200.0   Sylhet     age_spoof  
3          1200.0   Sylhet  amount_mimic  
4          1200.0   Sylhet   https_added  


**Dataset(Transaction Patterns normal bank transaction data)**

In [3]:
import pandas as pd
df = pd.read_csv('../raw/bank_transactions_data_kaggle.csv')
print(df.head())

                                               URL TransactionID AccountID  \
0            https://stripe.com/time-daughter-care      TX000001   AC60409   
1  https://americanexpress.com/church-back-require      TX000002   AC78434   
2            https://stripe.com/draw-degree-simple      TX000003   AC76900   
3            https://paypal.com/respond-sound-view      TX000004   AC10607   
4           https://stripe.com/throughout-be-exist      TX000005   AC65179   

   TransactionAmount      TransactionDate TransactionType Location DeviceID  \
0            3473.15  2025-01-25 15:34:24          Credit  Barisal  D710824   
1            4736.36  2025-11-01 14:25:49          Credit    Dhaka  D468190   
2            3094.34  2024-02-21 11:16:01          Credit   Khulna  D327385   
3            4674.42  2025-09-10 12:50:45           Debit   Sylhet  D932847   
4            1267.80  2025-04-14 23:35:19           Debit   Sylhet  D787060   

        IP Address MerchantID    Channel  CustomerAge Cu

In [14]:
import pandas as pd
import random

input_path = "../raw/bank_transactions_data_kaggle.csv"
output_path = "../processed/kaggle_legit.csv"

print("Loading Kaggle dataset...")
df = pd.read_csv(input_path)

df_small = df.head(1001)  
df_small["label"] = 0  

df_small.to_csv(output_path, index=False)

print("Saved Kaggle legitimate dataset to data/raw/kaggle_legit.csv")


Loading Kaggle dataset...
Saved Kaggle legitimate dataset to data/raw/kaggle_legit.csv


C:\Users\Sadrib\AppData\Local\Temp\ipykernel_10408\634665199.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small["label"] = 0


**Datasets(Transactions patterns phishing using phishtank )**

In [6]:
import pandas as pd
df = pd.read_csv('../raw/phishtank_urls.csv')
print(df.head())

   Unnamed: 0   FirstName LastName         Dob Gender    UserId  Status  \
0         484    ramkaran    verma  08-10-1981      U  12681731       1   
1         922         jai    singh  13-06-1991      F  13362335       1   
2         838       rohit      NaN  30-03-1994      U  12645319       1   
3         488  radheshyam      sen  25-08-2004      U  13323017       1   
4         453      sanjay      NaN  27-08-1976      U  13005431       1   

                 Email      Mobile  ProgramId  ... PresentLoginTime  \
0  ramXXXXXX@gmail.com  9494637598       6019  ...         17:56:24   
1  jaiXXXXXX@gmail.com  9536512418       6019  ...         02:05:02   
2  rohXXXXXX@gmail.com  9118530715       6019  ...         07:04:38   
3  radXXXXXX@gmail.com  8326769637       6019  ...         17:42:08   
4  sanXXXXXX@gmail.com  8322574629       6019  ...         19:46:58   

         RegisteredOn  OS Type Reg Referral Code Reg Referral Prefix  \
0  10/19/2020   19:00  Android        fagas21884  

In [13]:
import pandas as pd
import os


input_path = "../raw/phishtank_urls.csv"
output_path = "../processed/phishing.csv"


os.makedirs(os.path.dirname(output_path), exist_ok=True)

print(f"Loading PhishTank data from: {input_path}")

try:
    
    df_phish = pd.read_csv(input_path)
    df_phish = df_phish.head(1001)  

    
    df_phish["label"] = 1

    
    df_phish.to_csv(output_path, index=False)

    print(f"Successfully processed {len(df_phish)} phishing URLs.")
    print(f"Saved processed phishing dataset to: {output_path}")

except FileNotFoundError:
    print(f"Error: The input file {input_path} was not found.")
    print("Please make sure you have run the data collection script (`collect_phishtank.py`) first.")
except Exception as e:
    print(f"An error occurred during processing: {e}")

Loading PhishTank data from: ../raw/phishtank_urls.csv
Successfully processed 1000 phishing URLs.
Saved processed phishing dataset to: ../processed/phishing.csv


**Data preprocessing** 
#
After we label all datasets This step it's about cleaning transforming and preparing data so it's high quality and ready for modeling
#

In [13]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import os
import sys

def ensure_columns(df, cols_to_keep):
    for col in cols_to_keep:
        if col not in df.columns:
            df[col] = None
    return df[cols_to_keep]

if __name__ == "__main__":
    phishing_path = "../processed/phishing.csv"
    kaggle_legit_path = "../processed/kaggle_legit.csv"
    adversarial_path = "../processed/sessions_with_adv.csv"
    output_path = "../processed/preprocessed_dataset.csv"

    try:
        phishing = pd.read_csv(phishing_path)
        kaggle_legit = pd.read_csv(kaggle_legit_path)
        adversarial = pd.read_csv(adversarial_path)
    except FileNotFoundError as e:
        print(f"Error: Required file not found: {e.args[0]}. Please ensure input files are in the current directory.")
        sys.exit(1)

    phishing["label"] = 1
    kaggle_legit["label"] = 0
    adversarial["label"] = 1

    phishing = phishing.rename(columns={'Reg Client IP': 'IP Address'})
    phishing['TransactionDuration'] = None
    phishing['notes'] = None
    phishing['BadTryCount'] = phishing['BadTryCount'].fillna(0)
    phishing['Mobile'] = phishing['Mobile'].astype(str).fillna('MISSING')
    phishing['TransactionAmount'] = None
    phishing['TransactionType'] = None
    phishing['CustomerAge'] = None
    phishing['AccountBalance'] = None
    phishing['Location'] = None

    kaggle_legit = kaggle_legit.rename(columns={
        'URL': 'url',
        'DeviceID': 'Device Id',
        'LoginAttempts': 'BadTryCount'
    })
    kaggle_legit['notes'] = None
    kaggle_legit['Mobile'] = None

    adversarial['BadTryCount'] = None
    adversarial['Device Id'] = None
    adversarial['IP Address'] = None
    adversarial['TransactionDuration'] = None
    adversarial['Mobile'] = None

    required_cols = [
        "url", "TransactionAmount", "TransactionType", "CustomerAge",
        "AccountBalance", "Location", "BadTryCount", "Device Id",
        "notes", "label", "IP Address", "TransactionDuration", "Mobile"
    ]

    phishing = ensure_columns(phishing, required_cols)
    kaggle_legit = ensure_columns(kaggle_legit, required_cols)
    adversarial = ensure_columns(adversarial, required_cols)

    df = pd.concat([
        phishing.assign(source="phishing"),
        kaggle_legit.assign(source="legit"),
        adversarial.assign(source="adversarial")
    ], ignore_index=True)

    df['url'].fillna('NON_URL_TRANSACTION_PLACEHOLDER', inplace=True)
    
    numeric_cols = ['TransactionAmount', 'CustomerAge', 'AccountBalance', 'BadTryCount', 'TransactionDuration']
    categorical_cols = ['TransactionType', 'Location', 'Device Id', 'IP Address', 'Mobile', 'notes']

    for col in numeric_cols:
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)

    for col in categorical_cols:
        if df[col].isnull().any():
            df[col].fillna('MISSING', inplace=True)

    scaler = MinMaxScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

    min_class_size = df['label'].value_counts().min()

    df_class_0 = df[df['label'] == 0].sample(min_class_size, random_state=42)
    df_class_1 = df[df['label'] == 1].sample(min_class_size, random_state=42)

    df_balanced = pd.concat([df_class_0, df_class_1], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_balanced.to_csv(output_path, index=False)

    print(f"Preprocessed and balanced dataset saved to: {output_path}")

Preprocessed and balanced dataset saved to: ../processed/preprocessed_dataset.csv


C:\Users\Sadrib\AppData\Local\Temp\ipykernel_1784\2775740571.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([
C:\Users\Sadrib\AppData\Local\Temp\ipykernel_1784\2775740571.py:71: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['url'].fillna('NON_URL_TRANSACTION

**Data processing Output**
# 
Merged and processed labelled data of transactions patterns, adverserial and fake detections

In [14]:
import pandas as pd
df = pd.read_csv('../processed/preprocessed_dataset.csv')
print(df)


                                        url  TransactionAmount  \
0                         http://skrill.com           0.000000   
1     https://paypal.com/piece-sense-nation           0.380717   
2                        https://paypal.com           0.000900   
3                https://paytm.com/standard           0.703694   
4           NON_URL_TRANSACTION_PLACEHOLDER           0.000900   
...                                     ...                ...   
1995        NON_URL_TRANSACTION_PLACEHOLDER           0.000900   
1996                   http://apple pay.com           0.001901   
1997         https://paytm.com/medical-none           0.244795   
1998                   http://apple pay.com           0.000300   
1999              https://bankofamerica.com           0.001901   

     TransactionType  CustomerAge  AccountBalance    Location  BadTryCount  \
0              Debit     0.017544        0.000000  Chittagong          0.1   
1              Debit     0.789474        0.177499  

**Feature Engineering**
# 
This section creates advanced, robust features (like character entropy, domain length, etc.) from the combined dataset, preparing it for adverserial resistant and detecting fake payment detection approaches to the selected model.


In [38]:
import pandas as pd
from urllib.parse import urlparse
import numpy as np
import re

def calculate_entropy(s):
    if not isinstance(s, str) or not s:
        return 0.0
    freq = {}
    ent = 0.0
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    for v in freq.values():
        p = v / len(s)
        ent += -p * np.log2(p) if p > 0 else 0
    return ent

def extract_domain(url):
    try:
        parsed = urlparse(url)
        domain = parsed.netloc or parsed.path.split('/')[0] if parsed.path else ''
        domain = domain.split(':')[0]
        return domain
    except:
        return ''

def create_engineered_features(df):
    df_result = df.copy()
    
    df_result['url_length'] = df_result['url'].apply(len)
    df_result['path_length'] = df_result['url'].apply(lambda u: len(urlparse(u).path))
    df_result['ssl_flag'] = df_result['url'].apply(lambda u: 1 if str(u).startswith("https") else 0)
    
    df_result['domain'] = df_result['url'].apply(extract_domain)
    df_result['domain_entropy'] = df_result['domain'].apply(calculate_entropy)
    
    if 'DomainAge' in df_result.columns:
        df_result['DomainAge'] = pd.to_numeric(df_result['DomainAge'], errors='coerce')
        df_result['DomainAge'] = df_result['DomainAge'].fillna(df_result['DomainAge'].mean())
    else:
        df_result['DomainAge'] = 0
    
    if 'domain' not in df.columns:
        df_result = df_result.drop(columns=['domain'], errors='ignore')
    
    return df_result

if __name__ == "__main__":
    input_path = "../processed/preprocessed_dataset.csv"
    output_path = "../processed/Feature.csv"

    df = pd.read_csv(input_path)
    
    df_final = create_engineered_features(df)
    
    df_final.to_csv(output_path, index=False)
    print(f"Saved final engineered dataset to {output_path}")

Saved final engineered dataset to ../processed/Feature.csv


**Feature engineering output** 
#
Final output that is going to model the ultimate added featured

In [6]:
import pandas as pd
df = pd.read_csv('../processed/Feature.csv')
print(df)


                                        url  TransactionAmount  \
0                         http://skrill.com           0.000000   
1     https://paypal.com/piece-sense-nation           0.380717   
2                        https://paypal.com           0.000900   
3                https://paytm.com/standard           0.703694   
4           NON_URL_TRANSACTION_PLACEHOLDER           0.000900   
...                                     ...                ...   
1995        NON_URL_TRANSACTION_PLACEHOLDER           0.000900   
1996                   http://apple pay.com           0.001901   
1997         https://paytm.com/medical-none           0.244795   
1998                   http://apple pay.com           0.000300   
1999              https://bankofamerica.com           0.001901   

     TransactionType  CustomerAge  AccountBalance    Location  BadTryCount  \
0              Debit     0.017544        0.000000  Chittagong          0.1   
1              Debit     0.789474        0.177499  